# Generate an MCP PAT

Generates a cryptographically random personal access token for `MCP_PAT`, the bearer
token this server requires on every request.

`secrets.token_hex(32)` is the Python equivalent of `openssl rand -hex 32`: 32 random
bytes rendered as 64 hex characters, drawn from the OS CSPRNG.

The token is written to `.secrets/mcp_pat.txt`, which is gitignored so it cannot be
committed by accident.


> **Clear outputs before committing this notebook.**
>
> This notebook prints the token so you can copy it. `generate_mcp_pat.ipynb` is tracked
> by git, so if you run a cell and save the file, the token is stored in the notebook's
> output and would be committed in plain text. The token file itself is gitignored, but
> the notebook is not.
>
> Use *Clear All Outputs* in the notebook toolbar before staging any change here.


In [ ]:
import secrets
import stat
from pathlib import Path

# Set to True to replace a token that already exists.
FORCE = False

SECRET_DIR = Path(".secrets")
TOKEN_PATH = SECRET_DIR / "mcp_pat.txt"

SECRET_DIR.mkdir(mode=0o700, exist_ok=True)

if TOKEN_PATH.exists() and not FORCE:
    token = TOKEN_PATH.read_text().strip()
    print(f"{TOKEN_PATH} already exists, so it was left alone.")
    print("The existing token may already be deployed. Set FORCE = True to replace it.")
    print(f"\nCurrent token ({len(token)} chars):\n{token}")
else:
    # Equivalent to `openssl rand -hex 32`.
    token = secrets.token_hex(32)
    TOKEN_PATH.write_text(token + "\n")
    TOKEN_PATH.chmod(stat.S_IRUSR | stat.S_IWUSR)  # 0600, readable only by you
    print(f"Wrote a new token to {TOKEN_PATH} (mode 600).")
    print(f"\nToken ({len(token)} chars):\n{token}")

## Confirm the token cannot be committed

This should report that the file is ignored. If it does not, stop and fix `.gitignore`
before running any `git add`.


In [ ]:
import subprocess

check = subprocess.run(
    ["git", "check-ignore", "-v", str(TOKEN_PATH)],
    capture_output=True,
    text=True,
)
if check.returncode == 0:
    print(f"Safe: git ignores this file via {check.stdout.strip()}")
else:
    print("WARNING: this file is NOT gitignored. Add `.secrets/` to .gitignore now.")

## Rolling the token out

The token has to match in three places, or requests fail with `401`:

1. **GitHub** - repository *Settings -> Secrets and variables -> Actions*, set `MCP_PAT`.
   Secrets are write-only, so save the value somewhere you can read it back.
2. **The instance** - run *Actions -> Deploy ClinicalMCP to EC2 -> Run workflow*. The
   deploy rewrites `.env` on the box from the secret; changing the secret alone does
   nothing until a run happens.
3. **Your MCP client** - paste the same value at the `clinicalmcp_pat` prompt in
   `.vscode/mcp.json`, then run *MCP: Restart Server*.

Optionally update the local `.env` too, so local runs use the same token.

Verify the rollout worked (expects `200`):

```bash
TOKEN=$(cat .secrets/mcp_pat.txt)
curl -s -o /dev/null -w '%{http_code}\n' \
  http://ec2-18-191-206-22.us-east-2.compute.amazonaws.com/mcp \
  -X POST -H "Authorization: Bearer $TOKEN" \
  -H 'Content-Type: application/json' \
  -H 'Accept: application/json, text/event-stream' \
  -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-06-18","capabilities":{},"clientInfo":{"name":"check","version":"1"}}}'
```
